# Customer Behavior Analysis — Nâng cấp phân khúc khách hàng

**Mục tiêu:** kiểm tra dữ liệu, phân tích khám phá (EDA), dùng K-Means để phân nhóm khách hàng, kiểm định lựa chọn `k`, mô tả đặc điểm từng cụm và đề xuất thử nghiệm marketing.

**Nguồn:** `Mall_Customers.csv` trong repository `Huy054/customer-behavior-analysis`.

**Lưu ý nghiệp vụ:** `Spending Score` là điểm chi tiêu do bộ dữ liệu cung cấp, **không phải doanh thu bằng tiền**. Không thể khẳng định nhóm nào mang lại doanh thu lớn nhất, lợi nhuận cao nhất hoặc ROI tốt nhất từ các cột hiện có.

Chạy các ô theo thứ tự từ trên xuống. Notebook không xóa hoặc ghi đè dữ liệu gốc.

## 1. Chuẩn bị môi trường và đọc dữ liệu

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.optimize import linear_sum_assignment

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

candidates = [
    Path('Mall_Customers.csv'),
    Path('/content/Mall_Customers.csv'),
    Path('data/Mall_Customers.csv'),
    Path('../data/Mall_Customers.csv'),
]
input_file = next((p for p in candidates if p.is_file()), None)
if input_file is None:
    raise FileNotFoundError(
        'Không tìm thấy Mall_Customers.csv. '
        'hoặc đặt file trong thư mục data/ của repository.'
    )

df = pd.read_csv(input_file)
print('Đã đọc:', input_file.resolve())
print('Kích thước:', df.shape)
display(df.head())

## 2. Kiểm tra chất lượng dữ liệu

Không sử dụng `dropna()` một cách tự động; kiểm tra rồi mới quyết định cách xử lý.

In [ ]:
required = ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f'Thiếu cột bắt buộc: {missing_cols}')

for col in ['CustomerID', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Số dòng:', len(df))
print('Giá trị thiếu theo cột:')
print(df[required].isna().sum().to_string())
print('CustomerID bị trùng:', df['CustomerID'].duplicated().sum())
print('Dòng trùng hoàn toàn:', df.duplicated().sum())
print('Tuổi ngoài 0–120:', (~df['Age'].between(1, 120)).sum())
print('Thu nhập âm:', (df['Annual Income (k$)'] < 0).sum())
print('Spending Score ngoài 1–100:', (~df['Spending Score (1-100)'].between(1, 100)).sum())

if df[required].isna().any().any():
    raise ValueError('Dữ liệu có giá trị thiếu/không chuyển được kiểu. Hãy kiểm tra trước khi phân cụm.')
if df['CustomerID'].duplicated().any():
    raise ValueError('CustomerID bị trùng; cần kiểm tra dữ liệu trước khi tiếp tục.')
if not df['Age'].between(1,120).all() or not df['Annual Income (k$)'].ge(0).all() or not df['Spending Score (1-100)'].between(1,100).all():
    raise ValueError('Dữ liệu ngoài phạm vi dự kiến. Hãy xem lại các dòng bất thường.')

display(df[required].describe(include='all'))

## 3. EDA — mô tả khách hàng

Giới tính và tuổi được dùng để **mô tả kết quả**, không phải là biến đầu vào của mô hình K-Means trong bản phân tích này.

In [ ]:
print('Số khách theo giới tính:')
display(df['Gender'].value_counts().rename_axis('Gender').reset_index(name='Customers'))
print('Tuổi trung bình:', round(df['Age'].mean(), 2))
print('Thu nhập trung bình (k$):', round(df['Annual Income (k$)'].mean(), 2))
print('Spending Score trung bình:', round(df['Spending Score (1-100)'].mean(), 2))

plt.figure(figsize=(8, 5))
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], alpha=.7, s=38)
plt.xlabel('Thu nhập hàng năm (nghìn USD)')
plt.ylabel('Điểm chi tiêu (1–100)')
plt.title('Phân bố khách hàng trước phân cụm')
plt.grid(alpha=.18)
plt.show()

## 4. Chuẩn hóa và đánh giá k

Vì thu nhập và điểm chi tiêu có thang đo khác nhau, sử dụng `StandardScaler` để cả hai đặc trưng cùng tham gia vào phép tính khoảng cách. Kiểm tra `k = 2…10` bằng **Elbow (Inertia)** và **Silhouette Score**. Điểm Silhouette cao hơn thường biểu thị sự phân tách cụm tốt hơn, nhưng vẫn phải cân nhắc ý nghĩa kinh doanh.

In [ ]:
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

evaluations = []
for k in range(2, 11):
    candidate = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = candidate.fit_predict(X_scaled)
    evaluations.append({
        'k': k,
        'Inertia': candidate.inertia_,
        'Silhouette Score': silhouette_score(X_scaled, labels),
    })
eval_df = pd.DataFrame(evaluations)
display(eval_df.round(4))
print('k có Silhouette Score cao nhất trong khoảng thử:',
      int(eval_df.loc[eval_df['Silhouette Score'].idxmax(), 'k']))

plt.figure(figsize=(8, 4))
plt.plot(eval_df['k'], eval_df['Inertia'], marker='o')
plt.xticks(eval_df['k'])
plt.xlabel('Số cụm k')
plt.ylabel('Inertia')
plt.title('Elbow — độ phân tán nội cụm')
plt.grid(alpha=.2)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(eval_df['k'], eval_df['Silhouette Score'], marker='o')
plt.xticks(eval_df['k'])
plt.xlabel('Số cụm k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette theo số cụm')
plt.grid(alpha=.2)
plt.show()

## 5. Chọn và huấn luyện mô hình

Đặt **`K_SELECTED = 5`** tạo 5 nhóm marketing dễ diễn giải. Đây là **quyết định phân tích cần kiểm chứng**, không mặc định 5 là số cụm tối ưu. Sau khi xem biểu đồ ở bước 4, có thể điều chỉnh và chạy lại từ ô này.

In [ ]:
K_SELECTED = 5
model = KMeans(n_clusters=K_SELECTED, n_init=20, random_state=42)
df['Cluster'] = model.fit_predict(X_scaled)
selected_score = silhouette_score(X_scaled, df['Cluster'])
print(f'Silhouette Score của k={K_SELECTED}: {selected_score:.4f}')
print('Số khách theo Cluster:')
print(df['Cluster'].value_counts().sort_index().to_string())

## 6. Gắn nhãn mô tả dựa trên tâm cụm — không hardcode Cluster 0/1/2/3/4

Số hiệu `Cluster` do thuật toán tạo không có ý nghĩa cố định. Với `k=5`, gán *nhãn mô tả* một-một cho 5 tâm cụm bằng cách ghép với các vị trí tham chiếu trên không gian đã chuẩn hóa. Luôn xem bảng tâm cụm để kiểm tra nhãn có phù hợp. Không gắn nhãn “VIP” hay “nguồn doanh thu chính” khi thiếu dữ liệu tiền chi tiêu thực tế.

In [ ]:
centroids_original = pd.DataFrame(
    scaler.inverse_transform(model.cluster_centers_),
    columns=features
)
centroids_original['Cluster'] = np.arange(K_SELECTED)

if K_SELECTED == 5:
    # Năm mẫu vị trí lý tưởng: thấp-thấp, thấp-cao, trung bình, cao-thấp, cao-cao
    label_names = [
        'Thu nhập thấp – Chi tiêu thấp',
        'Thu nhập thấp – Chi tiêu cao',
        'Thu nhập và chi tiêu trung bình',
        'Thu nhập cao – Chi tiêu thấp',
        'Thu nhập cao – Chi tiêu cao',
    ]
    targets = np.array([[-1, -1], [-1, 1], [0, 0], [1, -1], [1, 1]], dtype=float)
    cost = np.sum((model.cluster_centers_[:, None, :] - targets[None, :, :])**2, axis=2)
    center_idx, target_idx = linear_sum_assignment(cost)
    segment_map = dict(zip(center_idx, [label_names[j] for j in target_idx]))
else:
    segment_map = {i: f'Nhóm {i} — chưa gán mô tả' for i in range(K_SELECTED)}

df['Segment'] = df['Cluster'].map(segment_map)
centroids_original['Segment'] = centroids_original['Cluster'].map(segment_map)
print('Tâm cụm trên thang đo gốc:')
display(centroids_original.round(2).sort_values('Cluster'))

## 7. Thống kê hồ sơ nhóm khách hàng

In [ ]:
profile = (
    df.groupby(['Cluster','Segment'], as_index=False)
      .agg(
          Customers=('CustomerID', 'count'),
          Mean_Age=('Age','mean'),
          Mean_Income_kUSD=('Annual Income (k$)','mean'),
          Mean_Spending_Score=('Spending Score (1-100)','mean'),
          Median_Spending_Score=('Spending Score (1-100)','median'),
      )
)
profile['Share_Pct'] = profile['Customers'] / len(df) * 100
profile = profile[['Cluster','Segment','Customers','Share_Pct','Mean_Age','Mean_Income_kUSD','Mean_Spending_Score','Median_Spending_Score']]
display(profile.round(2).sort_values('Mean_Income_kUSD'))
assert profile['Customers'].sum() == len(df)
print('Số khách đã phân nhóm:', int(profile['Customers'].sum()))

## 8. Trực quan hóa phân cụm

In [ ]:
plt.figure(figsize=(10, 6))
colors = plt.cm.tab10(np.linspace(0, 1, max(K_SELECTED, 3)))
for i in range(K_SELECTED):
    subset = df[df['Cluster'] == i]
    plt.scatter(
        subset['Annual Income (k$)'], subset['Spending Score (1-100)'],
        s=45, alpha=.75, color=colors[i], label=f'{i}: {segment_map[i]}'
    )
plt.scatter(
    centroids_original['Annual Income (k$)'], centroids_original['Spending Score (1-100)'],
    marker='X', s=190, color='black', linewidths=1, label='Tâm cụm'
)
plt.xlabel('Thu nhập hàng năm (nghìn USD)')
plt.ylabel('Điểm chi tiêu (1–100)')
plt.title('K-Means — Phân nhóm khách hàng theo thu nhập và điểm chi tiêu')
plt.legend(loc='best', fontsize=8)
plt.grid(alpha=.15)
plt.tight_layout()
plt.show()

## 9. Kiểm tra độ ổn định của kết quả phân cụm

So sánh nhãn từ nhiều `random_state` bằng **Adjusted Rand Index (ARI)**: gần 1 nghĩa là cấu trúc cụm tương đồng. Đây là kiểm tra độ ổn định kỹ thuật, **không** xác nhận hiệu quả chiến dịch marketing.

In [ ]:
stability_rows = []
for seed in [7, 21, 42, 123]:
    other = KMeans(n_clusters=K_SELECTED, n_init=20, random_state=seed)
    other_labels = other.fit_predict(X_scaled)
    stability_rows.append({
        'random_state': seed,
        'ARI_vs_seed42': adjusted_rand_score(df['Cluster'], other_labels),
    })
display(pd.DataFrame(stability_rows).round(4))

## 10. Đề xuất marketing — giả thuyết cần thử nghiệm

- **Thu nhập cao – Điểm chi tiêu cao:** cân nhắc thử nghiệm đặc quyền thành viên; chưa thể gọi là nhóm mang lại doanh thu cao nhất.
- **Thu nhập cao – Điểm chi tiêu thấp:** nghiên cứu rào cản mua sắm, thử nghiệm gợi ý sản phẩm phù hợp.
- **Thu nhập thấp – Điểm chi tiêu cao:** cân nhắc ưu đãi phù hợp ngân sách; không suy đoán tình trạng tài chính cá nhân.
- **Thu nhập thấp – Điểm chi tiêu thấp:** tìm hiểu nhu cầu trước khi phân bổ ngân sách, không kết luận nhóm “ít giá trị”.
- **Nhóm trung bình:** thử nghiệm tích điểm/chương trình giữ chân và đo hiệu quả thực tế.

**Khi triển khai thật:** phải có kết quả mua hàng, giá trị giao dịch, tần suất, dữ liệu đồng ý tiếp thị và thử nghiệm A/B để đánh giá tác động. Các đề xuất trên chỉ là hướng thử nghiệm từ dữ liệu mẫu.

## 11. Xuất dữ liệu phân nhóm và kết quả đánh giá

Các file được sinh ra dưới `outputs/` khi chạy notebook, **không ghi đè CSV nguồn**. Có thể dùng bảng hồ sơ để viết README/Business Insights sau khi xác minh số liệu.

In [ ]:
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir/'Customer_Segmentation_Result.csv', index=False, encoding='utf-8-sig')
profile.to_csv(out_dir/'Cluster_Profile.csv', index=False, encoding='utf-8-sig')
eval_df.to_csv(out_dir/'KMeans_Evaluation.csv', index=False, encoding='utf-8-sig')
print('Đã xuất:')
for file in sorted(out_dir.glob('*.csv')):
    print('-', file, '|', file.stat().st_size, 'bytes')